# Character-level language modeling and decoding

A language model estimates a probability distribution for the next token from the tokens that came before it. This optional, CPU-only exercise makes that process concrete with characters as tokens: it trains a small LSTM to predict the next character in a passage from *Metamorphosis*, then turns the model's output scores into generated text.

You will inspect next-character training pairs, see how logits become probabilities, and compare greedy decoding, temperature-controlled sampling, top-k sampling, top-p sampling, and beam search. The fixed seed and deliberately short training run keep the example reproducible and quick to run. Its goal is to reveal how decoding choices affect the same model distribution, not to produce polished prose.


In [1]:
from pathlib import Path
from tempfile import gettempdir
from urllib.request import urlretrieve

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(7)
torch.set_num_threads(1)
CACHE_DIR = Path(gettempdir()) / 'genai_ch03_cache'
CACHE_DIR.mkdir(exist_ok=True)
TEXT_PATH = CACHE_DIR / 'metamorphosis_franz_kafka.txt'
if not TEXT_PATH.exists():
    print(f'Downloading public-domain text to {TEXT_PATH}')
    urlretrieve('https://www.gutenberg.org/cache/epub/5200/pg5200.txt', TEXT_PATH)
else:
    print(f'Using cached public-domain text at {TEXT_PATH}')
text = TEXT_PATH.read_text(encoding='utf-8')[:12_000]
characters = sorted(set(text))
stoi = {character: index for index, character in enumerate(characters)}
itos = dict(enumerate(characters))
encoded = torch.tensor([stoi[character] for character in text], dtype=torch.long)
print(f'characters in training excerpt: {len(text):,}; vocabulary size: {len(characters)}')


Using cached public-domain text at /var/folders/q6/nvytr45s0sn6b0_gm3rmxrdm0000gn/T/genai_ch03_cache/metamorphosis_franz_kafka.txt
characters in training excerpt: 12,000; vocabulary size: 73


## Build next-character examples

Language-model training converts one text sequence into many input-target pairs. Here each input is a 64-character context window and the target is the single character immediately after it. The model receives integer character IDs, uses an embedding layer to map them to vectors, and returns one score for every character in the vocabulary.


In [2]:
class CharacterDataset(Dataset):
    def __init__(self, values, context_length=64):
        self.values, self.context_length = values, context_length

    def __len__(self):
        return len(self.values) - self.context_length

    def __getitem__(self, index):
        context = self.values[index:index + self.context_length]
        next_character = self.values[index + self.context_length]
        return context, next_character

class CharacterLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_size=32, hidden_size=96):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.lstm = nn.LSTM(embedding_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, token_ids):
        states, _ = self.lstm(self.embedding(token_ids))
        return self.output(states[:, -1])


### Inspect one training pair

The printed target is the answer the model is asked to predict after the displayed context. During training, the same sliding-window construction supplies many such pairs from the text excerpt.


In [3]:
CONTEXT, BATCH_SIZE, EPOCHS = 64, 64, 2
dataset = CharacterDataset(encoded, CONTEXT)
example_context, example_target = dataset[0]
context_text = ''.join(itos[index.item()] for index in example_context)
print(f'context window: {context_text!r}')
print(f'next-character target: {itos[example_target.item()]!r}')


context window: 'The Project Gutenberg eBook of Metamorphosis\n    \nThis eBook is '
next-character target: 'f'


## Train a small language model

Cross-entropy compares the model's vocabulary-wide scores with the observed next character. The model is in training mode only while its parameters are updated; later cells switch to evaluation mode and inference-only execution for decoding.


In [4]:
loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(7),
)
model = CharacterLSTM(len(characters))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
loss_function = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for inputs, targets in loader:
        optimizer.zero_grad()
        loss = loss_function(model(inputs), targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'epoch {epoch + 1}: loss={total_loss / len(loader):.3f}')


epoch 1: loss=2.781
epoch 2: loss=2.342


## Inspect the next-character distribution

For one context, the LSTM emits unnormalized scores called logits, one per possible next character. Softmax converts those scores into probabilities that sum to one. Greedy decoding chooses the highest-probability character, while the following decoding strategies choose from this same distribution in different ways.


In [5]:
model.eval()
with torch.inference_mode():
    example_logits = model(example_context.unsqueeze(0))[0]
    example_probabilities = torch.softmax(example_logits, dim=-1)
    selected_id = example_probabilities.argmax().item()
    top_probabilities, top_ids = torch.topk(example_probabilities, 5)

print(f'logits shape: {tuple(example_logits.shape)}')
print('top softmax probabilities:', [(itos[index.item()], round(probability.item(), 3)) for probability, index in zip(top_probabilities, top_ids)])
print(f'selected character (greedy argmax): {itos[selected_id]!r}')


logits shape: (73,)
top softmax probabilities: [('a', 0.208), ('t', 0.168), ('h', 0.098), ('o', 0.063), ('i', 0.055)]
selected character (greedy argmax): 'a'


## Decode the distribution

Sampling draws a character according to the probabilities rather than always selecting the largest one. Temperature rescales logits before softmax: lower values concentrate probability on the most likely characters, while higher values spread it more broadly. Top-k samples only from the *k* highest-scoring characters; top-p samples from the smallest set whose cumulative probability reaches *p*. Each strategy then appends its selected character to the context and repeats.


In [6]:
def sample_next(logits, temperature=1.0, top_k=None, top_p=None, generator=None):
    if temperature <= 0:
        raise ValueError('temperature must be positive')
    if top_k is not None and top_k < 1:
        raise ValueError('top_k must be at least 1')
    if top_p is not None and not 0 < top_p <= 1:
        raise ValueError('top_p must satisfy 0 < top_p <= 1')

    filtered_logits = logits / temperature
    if top_k is not None:
        cutoff = torch.topk(filtered_logits, min(top_k, filtered_logits.numel())).values[-1]
        filtered_logits = filtered_logits.masked_fill(filtered_logits < cutoff, float('-inf'))
    if top_p is not None:
        ordered_logits, ordered_indices = torch.sort(filtered_logits, descending=True)
        cumulative_probabilities = torch.softmax(ordered_logits, dim=-1).cumsum(dim=-1)
        remove = cumulative_probabilities > top_p
        remove[1:] = remove[:-1].clone()
        remove[0] = False
        filtered_logits = filtered_logits.scatter(0, ordered_indices[remove], float('-inf'))

    probabilities = torch.softmax(filtered_logits, dim=-1)
    return torch.multinomial(probabilities, 1, generator=generator).item()

@torch.inference_mode()
def generate(prompt, length=120, strategy='sample', temperature=0.8, top_k=None, top_p=None):
    if strategy not in {'greedy', 'sample'}:
        raise ValueError("strategy must be 'greedy' or 'sample'")
    ids = [stoi.get(character, stoi[' ']) for character in prompt]
    generator = torch.Generator().manual_seed(7)
    model.eval()
    for _ in range(length):
        logits = model(torch.tensor(ids[-CONTEXT:], dtype=torch.long).unsqueeze(0))[0]
        next_id = logits.argmax().item() if strategy == 'greedy' else sample_next(
            logits, temperature, top_k, top_p, generator
)
        ids.append(next_id)
    return ''.join(itos[index] for index in ids)


### Compare decoding choices

All runs start from the same prompt and trained model. The sampling generator is reset to the same seed for each run, so differences in the printed continuations reflect the selected decoding settings rather than an uncontrolled random draw.


In [7]:
prompt = 'Gregor '
decoding_runs = {
    'greedy': {'strategy': 'greedy'},
    'lower temperature (0.4)': {'temperature': 0.4},
    'higher temperature (1.2)': {'temperature': 1.2},
    'top-k (k=8, temperature=0.8)': {'temperature': 0.8, 'top_k': 8},
    'top-p (p=0.9, temperature=0.8)': {'temperature': 0.8, 'top_p': 0.9},
}
for name, kwargs in decoding_runs.items():
    print(f'\n{name}:\n{generate(prompt, **kwargs)}')



greedy:
Gregor the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the 

lower temperature (0.4):
Gregor his it to the and at the to fo and and he mace ang on an he thould the the wat and and of and and and the and the wave t

higher temperature (1.2):
Gregor hile
the Prjdanemagt thl pomeciane tnaiveve cy
geg ont
upeessioweshee
curel int in bengerfsik pifB o2out avitut jas vave

top-k (k=8, temperature=0.8):
Gregor his it to to ane ant the coull and thatheve cougely
ther he sis
winterere
of he to therer ant all ingout avit there and 

top-p (p=0.9, temperature=0.8):
Gregor his it to to and ant the poull and thaind mac ande ont the could the and allint in to ger ant alf ingout ave at and and 


## Keep several candidate continuations with beam search

Beam search does not sample. At each step, it expands several high-scoring partial continuations, adds their log probabilities, and keeps only the best few complete candidates. This lets it consider more than the locally best next character, but it can still favor repetitive continuations.


In [8]:
@torch.inference_mode()
def beam_search(prompt, length=120, beam_width=3):
    initial = [stoi.get(character, stoi[' ']) for character in prompt]
    beams = [(initial, 0.0)]
    model.eval()
    for _ in range(length):
        candidates = []
        for token_ids, log_score in beams:
            logits = model(torch.tensor(token_ids[-CONTEXT:], dtype=torch.long).unsqueeze(0))[0]
            values, indices = torch.topk(torch.log_softmax(logits, dim=-1), beam_width)
            candidates.extend(
                (token_ids + [index.item()], log_score + value.item())
                for value, index in zip(values, indices)
)
        beams = sorted(candidates, key=lambda item: item[1], reverse=True)[:beam_width]
    return ''.join(itos[index] for index in beams[0][0])

print('beam search:\n', beam_search(prompt, beam_width=3))


beam search:
 Gregor the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the 


This fixed-length character example accumulates **log probabilities** across complete beams. Real variable-length beam search also needs an EOS token and length normalization. For open-ended text, compare its often repetitive output against top-k and top-p sampling rather than assuming it is better. The limited output quality here reflects the intentionally small training exercise.
